# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

Pre bilo kakvog testiranja signala, gledamo raspodele ključnih kolona koje planiramo da koristimo:
impressions_90d, ctr, avg_position, word_count, engagement_rate. Web/traffic metrike su skoro 
uvek heavy-tailed (par giganata, dugačak rep sitnih vrednosti) — to menja kako treba da se 
koreliše i grupiše dalje u analizi.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

repo_root = next(
    parent for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "data" / "raw" / "content_refresh_anonymized.csv").exists()
)
df = pd.read_csv(repo_root / "data" / "raw" / "content_refresh_anonymized.csv")
df["is_declining"] = (df["trend_direction"].str.lower() == "down").astype(int)

key_columns = ["impressions_90d", "ctr", "avg_position", "word_count", "engagement_rate", "search_volume"]

print("Osnovna statistika (uočiti heavy tail: mean >> median):")
print(df[key_columns].describe().round(2))

print("\nProcenat nula/blanko po koloni:")
for col in key_columns:
    zero_pct = (df[col].fillna(0) == 0).mean()
    print(f"  {col}: {zero_pct:.1%} je 0 ili prazno")

Osnovna statistika (uočiti heavy tail: mean >> median):
       impressions_90d       ctr  avg_position  word_count  engagement_rate  \
count         30000.00  30000.00      30000.00    22301.00         30000.00   
mean           5200.37      0.51         16.34     3107.76             2.53   
std           16838.02      3.28         15.22     1452.38             8.31   
min               1.00      0.00          0.00        8.00             0.00   
25%              81.00      0.00          6.20     2413.00             0.00   
50%             731.00      0.07         10.80     2877.00             0.00   
75%            3615.25      0.29         22.30     3666.00             1.35   
max          517715.00    100.00        245.00     9546.00           100.00   

       search_volume  
count       27532.00  
mean          158.88  
std          1518.27  
min             0.00  
25%             0.00  
50%            10.00  
75%            20.00  
max         74000.00  

Procenat nula/blanko po 

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

Tri signala, svaki sa mini-testom. Prva dva su iz mog baseline rada (Nedelja 4), 
treći je nov test popularnog verovanja iz mog lane-a.

In [2]:
df["recent_share"] = df["impressions_last_30d"] / df["impressions_90d"]
df["recent_share_bucket"] = np.where(df["recent_share"] < 0.30, "under_30pct", "30pct_or_more")

signal1 = df.groupby("recent_share_bucket").agg(
    declining_rate=("is_declining", "mean"),
    n=("is_declining", "size"),
).round(3)
print("Signal 1: recent_share vs is_declining")
print(signal1)

# Hipoteza: duži sadržaj opada ređe (ima "više supstance" da zadrži rangiranje)
wc_test = df[df["word_count"] > 0].copy()  # izbaci redove bez podataka o word_count

wc_test["word_count_bucket"] = pd.cut(
    wc_test["word_count"],
    bins=[0, 1000, 2000, 3500, float("inf")],
    labels=["<1000", "1000-2000", "2000-3500", "3500+"],
)

signal2 = wc_test.groupby("word_count_bucket", observed=True).agg(
    declining_rate=("is_declining", "mean"),
    n=("is_declining", "size"),
).round(3)
print("Signal 2: word_count vs is_declining")
print(signal2)

# Hipoteza: stranice sa transakcionim intentom imaju viši CTR (ljudi spremni da kliknu kad kupuju)
visible = df[df["impressions_90d"] >= 100]

signal3 = visible.groupby("main_intent").agg(
    mean_ctr=("ctr", "mean"),
    n=("ctr", "size"),
).sort_values("mean_ctr", ascending=False).round(4)
print("Signal 3: mean CTR by main_intent")
print(signal3)

Signal 1: recent_share vs is_declining
                     declining_rate      n
recent_share_bucket                       
30pct_or_more                 0.081   9494
under_30pct                   0.755  20506
Signal 2: word_count vs is_declining
                   declining_rate      n
word_count_bucket                       
<1000                       0.207    973
1000-2000                   0.555   3781
2000-3500                   0.588  11262
3500+                       0.597   6285
Signal 3: mean CTR by main_intent
               mean_ctr      n
main_intent                   
navigational     0.3187     23
transactional    0.2759   4603
commercial       0.2484   3676
informational    0.2405  13210


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

Testiram signal iza "quick-win" flega sa predavanja: ideja je da visok search_volume za ciljanu 
ključnu reč znači da postoji "neiskorišćena tražnja" koju stranica treba samo malo da gurne da 
bi je uhvatila. Proveravam da li visok search_volume STVARNO predviđa veći stvarni saobraćaj.

In [3]:
volume_test = df[df["search_volume"].notna() & (df["search_volume"] > 0)].copy()

volume_test["volume_bucket"] = pd.qcut(
    volume_test["search_volume"], q=4, duplicates="drop"
)

signal_flag = volume_test.groupby("volume_bucket", observed=True).agg(
    mean_impressions=("impressions_90d", "mean"),
    mean_clicks=("clicks_90d", "mean"),
    n=("search_volume", "size"),
).round(1)
print("Flag-linked signal: search_volume (quartile) vs actual traffic")
print(signal_flag)

correlation = volume_test["search_volume"].corr(volume_test["impressions_90d"])
print(f"\nKorelacija search_volume vs impressions_90d: {correlation:.3f}")

Flag-linked signal: search_volume (quartile) vs actual traffic
                 mean_impressions  mean_clicks     n
volume_bucket                                       
(9.999, 20.0]              5153.6         17.5  9601
(20.0, 70.0]               5950.3         16.9  3339
(70.0, 74000.0]            5649.9         13.5  3511

Korelacija search_volume vs impressions_90d: 0.003


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

Za content tim, tri nalaza iz ove nedelje:

1. Recent_share je i dalje najpouzdaniji brzi indikator opadanja koji imam (potvrđeno i u baseline 
   notebook-u) — ali pazi, mehanički je blizak samoj definiciji trenda, pa ne otkriva ništa novo 
   nezavisno.

2. Dužina sadržaja (word_count) NE predviđa pouzdano da li stranica opada — trebalo bi prestati 
   sa savetom "produži članak" kao generičkim refresh predlogom; treba tražiti specifičniji razlog 
   (CTR, poziciju, engagement) pre nego što se preporuči prosto dodavanje teksta.

3. search_volume iz keyword alata ne predviđa pouzdano stvarni saobraćaj — "quick-win" flegovi 
   zasnovani samo na visokom search_volume-u treba da se kombinuju sa stvarnim signalima vidljivosti 
   (impressions_90d, position_tier), ne da se koriste sami.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.